# Figure 1(B): Spectral Conflict and Task Discriminability

This notebook reproduces Figure 1(B):
- X-axis: Singular Value Percentile Bands (0-20%, 20-50%, 50-100%)
- Y-axis: Dual bar chart showing Conflict % and SV Importance %

**Key Observation**: High-SV components contain most conflicts AND most task-discriminative info.

In [ ]:
import sys
sys.path.insert(0, '../..')

import json
import os
import re
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Configuration

In [ ]:
PROJECT_ROOT = '.'
BASE_MODEL_PATH = f'{PROJECT_ROOT}/data/llama-2-7b'
DATASET_DIR = f'{PROJECT_ROOT}/data/flan_v2_subset'
CHECKPOINT_PATH = f'{PROJECT_ROOT}/output/bbh/2_fgr_granularity/2a_component/g4096_scalar/sft_lora_model'
SV_BINS = [0, 20, 50, 100]
HIGHLIGHT_REGION = (0, 20)

print('=' * 60)
print('Checkpoint Path Verification:')
print('=' * 60)
ckpt_path = Path(CHECKPOINT_PATH)
status = 'OK' if ckpt_path.exists() else 'NOT FOUND'
print(f'[{status}] {CHECKPOINT_PATH}')
print('=' * 60)

## 2. Extract B Matrices from Checkpoint

In [ ]:
def extract_b_matrices_from_state_dict(checkpoint_path: str) -> Dict[str, List[torch.Tensor]]:
    adapter_path = Path(checkpoint_path) / 'adapter_model.bin'
    if not adapter_path.exists():
        adapter_path = Path(checkpoint_path) / 'adapter_model.safetensors'
    print(f'Loading adapter from {adapter_path}...')
    if str(adapter_path).endswith('.bin'):
        state_dict = torch.load(adapter_path, map_location='cpu')
    else:
        from safetensors.torch import load_file
        state_dict = load_file(adapter_path)
    b_matrices = defaultdict(list)
    b_pattern = re.compile(r'^(.+)\.lora_B(\d+)\.weight$')
    for key, value in state_dict.items():
        match = b_pattern.match(key)
        if match:
            layer_name = match.group(1)
            expert_idx = int(match.group(2))
            b_matrices[layer_name].append((expert_idx, value.clone()))
    sorted_b_matrices = {}
    for layer_name, expert_list in b_matrices.items():
        sorted_list = sorted(expert_list, key=lambda x: x[0])
        sorted_b_matrices[layer_name] = [v for _, v in sorted_list]
    print(f'\nExtracted B matrices from {len(sorted_b_matrices)} layers')
    return sorted_b_matrices

b_matrices = extract_b_matrices_from_state_dict(CHECKPOINT_PATH)

## 3. SVD and Spectral Analysis Functions

In [ ]:
def compute_svd(b_matrix: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    b_float = b_matrix.float()
    U, S, Vh = torch.linalg.svd(b_float, full_matrices=False)
    return U, S, Vh

def compute_spectral_conflict_score(b_matrices_list: List[torch.Tensor], sv_percentile_range: Tuple[int, int]) -> float:
    N = len(b_matrices_list)
    if N < 2:
        return 0.0
    svd_results = [compute_svd(b) for b in b_matrices_list]
    rank = svd_results[0][1].shape[0]
    start_idx = int(rank * sv_percentile_range[0] / 100)
    end_idx = int(rank * sv_percentile_range[1] / 100)
    if start_idx >= end_idx:
        return 0.0
    band_size = end_idx - start_idx
    total_conflict = 0.0
    for k in range(start_idx, end_idx):
        for i in range(N):
            for j in range(N):
                if i != j:
                    sigma_i = svd_results[i][1][k].item()
                    sigma_j = svd_results[j][1][k].item()
                    u_i = svd_results[i][0][:, k]
                    u_j = svd_results[j][0][:, k]
                    cos_sim = torch.abs(F.cosine_similarity(u_i.unsqueeze(0), u_j.unsqueeze(0))).item()
                    total_conflict += sigma_i * sigma_j * cos_sim
    return total_conflict / (band_size * N * (N - 1))

def compute_sv_importance(b_matrices_list: List[torch.Tensor], sv_percentile_range: Tuple[int, int]) -> float:
    N = len(b_matrices_list)
    if N < 1:
        return 0.0
    svd_results = [compute_svd(b) for b in b_matrices_list]
    rank = svd_results[0][1].shape[0]
    start_idx = int(rank * sv_percentile_range[0] / 100)
    end_idx = int(rank * sv_percentile_range[1] / 100)
    if start_idx >= end_idx:
        return 0.0
    svi_list = []
    for U, S, Vh in svd_results:
        band_sv_sum = S[start_idx:end_idx].sum().item()
        total_sv_sum = S.sum().item()
        svi = band_sv_sum / max(total_sv_sum, 1e-10) * 100
        svi_list.append(svi)
    return np.mean(svi_list)

## 4. Compute Metrics Across Spectral Bands

In [ ]:
def analyze_spectral_bands(b_matrices: Dict[str, List[torch.Tensor]], bins: List[int]) -> Tuple[Dict[str, List[float]], Dict[str, List[float]]]:
    all_conflict_scores = defaultdict(list)
    all_svi_scores = defaultdict(list)
    bands = [(bins[i], bins[i + 1]) for i in range(len(bins) - 1)]
    print(f'Analyzing {len(bands)} spectral bands across {len(b_matrices)} layers...')
    for layer_name, b_list in tqdm(b_matrices.items()):
        if len(b_list) < 2:
            continue
        for band in bands:
            band_key = f'{band[0]}-{band[1]}%'
            conflict = compute_spectral_conflict_score(b_list, band)
            all_conflict_scores[band_key].append(conflict)
            svi = compute_sv_importance(b_list, band)
            all_svi_scores[band_key].append(svi)
    return all_conflict_scores, all_svi_scores

conflict_scores, svi_scores = analyze_spectral_bands(b_matrices, SV_BINS)

In [ ]:
def aggregate_results(conflict_scores, svi_scores):
    mean_conflict = {}
    mean_svi = {}
    for band_key in conflict_scores.keys():
        mean_conflict[band_key] = np.mean(conflict_scores[band_key])
        mean_svi[band_key] = np.mean(svi_scores[band_key])
    total_conflict = sum(mean_conflict.values())
    conflict_percentage = {k: v / total_conflict * 100 if total_conflict > 0 else 0 for k, v in mean_conflict.items()}
    return mean_conflict, mean_svi, conflict_percentage

mean_conflict, mean_svi, conflict_percentage = aggregate_results(conflict_scores, svi_scores)

print('\n' + '=' * 80)
print('Spectral Analysis Results')
print('=' * 80)
print(f'{"Band":<15} {"Conflict %":<15} {"SV Importance %":<20}')
print('-' * 80)
for band_key in sorted(mean_conflict.keys(), key=lambda x: int(x.split('-')[0])):
    print(f'{band_key:<15} {conflict_percentage[band_key]:<15.2f} {mean_svi[band_key]:<20.2f}')
print('=' * 80)

## 5. Plot Figure 1(B)

In [ ]:
bands_sorted = sorted(mean_conflict.keys(), key=lambda x: int(x.split('-')[0]))
x_labels = [b.replace('%', '') for b in bands_sorted]
x = np.arange(len(bands_sorted))
width = 0.35
conflicts = [conflict_percentage[b] for b in bands_sorted]
svi_values = [mean_svi[b] for b in bands_sorted]

fig, ax = plt.subplots(figsize=(10, 6))
color_conflict = '#e74c3c'
color_svi = '#3498db'
bars1 = ax.bar(x - width / 2, conflicts, width, label='Conflict (%)', color=color_conflict, alpha=0.8)
bars2 = ax.bar(x + width / 2, svi_values, width, label='SV Importance (%)', color=color_svi, alpha=0.8)

ax.set_xlabel('Singular Value Percentile Band', fontsize=12)
ax.set_ylabel('Percentage (%)', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(x_labels, fontsize=11)
ax.legend(fontsize=11)

for bar, val in zip(bars1, conflicts):
    ax.annotate(f'{val:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()), xytext=(0, 3), textcoords='offset points', ha='center', fontsize=10, color=color_conflict)
for bar, val in zip(bars2, svi_values):
    ax.annotate(f'{val:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()), xytext=(0, 3), textcoords='offset points', ha='center', fontsize=10, color=color_svi)

ax.axvspan(-0.5, 0.5, alpha=0.15, color='yellow', zorder=0)
plt.title('Figure 1(B): Spectral Conflict vs Task Discriminability', fontsize=13, fontweight='bold')
plt.tight_layout()
os.makedirs('output', exist_ok=True)
plt.savefig('output/fig1b_spectral_conflict.png', dpi=300, bbox_inches='tight')
plt.show()
print('\nFigure saved to output/fig1b_spectral_conflict.png')

## 6. Summary

In [ ]:
import pandas as pd
df = pd.DataFrame({'SV Band': bands_sorted, 'Conflict %': [conflict_percentage[b] for b in bands_sorted], 'SV Importance %': [mean_svi[b] for b in bands_sorted]})

print('\n' + '=' * 70)
print('Figure 1(B) Results Summary')
print('=' * 70)
print(df.to_string(index=False))
print('=' * 70)

high_sv_conflict = conflict_percentage.get('0-20%', 0)
high_sv_svi = mean_svi.get('0-20%', 0)
low_sv_conflict = conflict_percentage.get('50-100%', 0)
low_sv_svi = mean_svi.get('50-100%', 0)

print(f'\nHigh-SV (0-20%): Conflict={high_sv_conflict:.1f}%, SV Importance={high_sv_svi:.1f}%')
print(f'Low-SV (50-100%): Conflict={low_sv_conflict:.1f}%, SV Importance={low_sv_svi:.1f}%')
print('\n=> High-SV region contains most conflicts AND most task-discriminative info.')
print('=> Cannot simply remove high-SV components to reduce conflict.')
print('=' * 70)